In [1]:
import sys
sys.path.insert(0, "/home/jovyan/work")

import pandas as pd
from src.features import add_hour_of_day, split_train_val_test, to_arrays
from src.train import train_from_csv

df = pd.read_csv("/home/jovyan/data/creditcard.csv")
df_fe = add_hour_of_day(df)

print(df_fe[["Time", "hour_of_day", "Amount", "Class"]].head())

   Time  hour_of_day  Amount  Class
0   0.0     0.000000  149.62      0
1   0.0     0.000000    2.69      0
2   1.0     0.000278  378.66      0
3   1.0     0.000278  123.50      0
4   2.0     0.000556   69.99      0


In [2]:
train_df, val_df, test_df = split_train_val_test(df_fe)

print(
    f"Train: {len(train_df)} | "
    f"Val: {len(val_df)} | "
    f"Test: {len(test_df)}"
)

Train: 170883 | Val: 56962 | Test: 56962


In [3]:
from pathlib import Path

modelo_v1 = Path("/home/jovyan/work/models/model.ubj")

if not modelo_v1.exists():
    result_v1 = train_from_csv(
        "/home/jovyan/data/creditcard.csv",
        eta=0.1,
        max_depth=5,
        n_estimators=200,
        model_out=str(modelo_v1),
    )
    print("V1 VAL:", result_v1.val_metrics)
    print("V1 TEST:", result_v1.test_metrics)
else:
    print("Modelo V1 ja existe. Preservado.")

val_pr_auc=0.8204
V1 VAL: {'precision': 0.9047619047619048, 'recall': 0.7676767676767676, 'f1': 0.8306010928961749, 'pr_auc': 0.8203686668034081}
V1 TEST: {'precision': 0.8791208791208791, 'recall': 0.8163265306122449, 'f1': 0.8465608465608465, 'pr_auc': 0.8761851545168435}


In [4]:
modelo_v2 = Path("/home/jovyan/work/models/model-v2.ubj")

if not modelo_v2.exists():
    result_v2 = train_from_csv(
        "/home/jovyan/data/creditcard.csv",
        eta=0.2,
        max_depth=4,
        n_estimators=200,
        model_out=str(modelo_v2),
    )
    print("V2 VAL:", result_v2.val_metrics)
    print("V2 TEST:", result_v2.test_metrics)
else:
    print("Modelo V2 ja existe. Preservado.")

val_pr_auc=0.8223
V2 VAL: {'precision': 0.9176470588235294, 'recall': 0.7878787878787878, 'f1': 0.8478260869565217, 'pr_auc': 0.8223320394996089}
V2 TEST: {'precision': 0.8791208791208791, 'recall': 0.8163265306122449, 'f1': 0.8465608465608465, 'pr_auc': 0.8653748192578966}


In [5]:
if "result_v1" in globals() and "result_v2" in globals():
    comparison = pd.DataFrame({
        "modelo_v1 (val)": result_v1.val_metrics,
        "modelo_v2 (val)": result_v2.val_metrics,
        "modelo_v1 (test)": result_v1.test_metrics,
        "modelo_v2 (test)": result_v2.test_metrics,
    }).round(4)

    print(comparison)
else:
    print("Comparacao requer os resultados dos dois treinamentos.")

assert modelo_v1.is_file()
assert modelo_v2.is_file()

print("TREINO XGBOOST CONCLUIDO")

           modelo_v1 (val)  modelo_v2 (val)  modelo_v1 (test)  \
precision           0.9048           0.9176            0.8791   
recall              0.7677           0.7879            0.8163   
f1                  0.8306           0.8478            0.8466   
pr_auc              0.8204           0.8223            0.8762   

           modelo_v2 (test)  
precision            0.8791  
recall               0.8163  
f1                   0.8466  
pr_auc               0.8654  
TREINO XGBOOST CONCLUIDO
